# Investigation: OVER After 3PT Make — DraftKings

**Trigger finding:** UNDER-after-3PT backtest hit only 41.5% (n=407, all books) → implied OVER rate 58.5%.  
**This notebook:** Uses only DraftKings (n=191 signals) to investigate whether that edge is real.

**Phases:**
- Phase 1: Foundational validity (real odds ROI, concentration, hold)
- Phase 2: Effect decomposition (jump magnitude, MC contrarian, hot-hand, quarter CIs)
- Phase 3: Out-of-sample validity (reg season vs playoffs, market adaptation)

All data already in S3 or ESPN API (same as backtest). No new infrastructure needed.

In [ ]:
import subprocess, warnings, time, re
import duckdb
import pandas as pd
import numpy as np
import requests
import pytz
from datetime import datetime, timezone, timedelta
from scipy import stats

def wilson_ci(k, n, alpha=0.10):
    """Wilson score interval."""
    z = stats.norm.ppf(1 - alpha/2)
    p = k/n if n > 0 else 0
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return center - margin, center + margin

warnings.filterwarnings('ignore')
ET = pytz.timezone('US/Eastern')

SESSION = requests.Session()
SESSION.verify = False
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'})

# ── parameters ──────────────────────────────────────────────────────────────
BOOKMAKER         = 'draftkings'
WINDOW_SECONDS    = 60
MIN_LINE_JUMP     = 0.5
MAX_POLL_INTERVAL = 65
EDGE_THRESHOLDS   = [0.05, 0.10, 0.15, 0.20]
EXCLUDED_BKS      = {'bovada', 'betonlineag'}

S3_BUCKET  = 'nba-betting-mt'
S3_SIGNALS = 'data/04_output/live_betting_signals/player_points'
print(f'Bookmaker: {BOOKMAKER}  |  window={WINDOW_SECONDS}s  |  max_interval={MAX_POLL_INTERVAL}s')

In [ ]:
def _get_cred(key):
    out = subprocess.run(['aws', 'configure', 'get', key], capture_output=True, text=True, timeout=5)
    return out.stdout.strip() if out.returncode == 0 else ''

AK = _get_cred('aws_access_key_id')
SK = _get_cred('aws_secret_access_key')
assert AK and SK, 'AWS credentials not found'

def duckdb_con():
    con = duckdb.connect(':memory:')
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.execute("SET s3_region='us-east-2';")
    con.execute(f"SET s3_access_key_id='{AK}';")
    con.execute(f"SET s3_secret_access_key='{SK}';")
    return con

print('AWS creds OK.')

## 1. Load Data

In [ ]:
# UNDER signals for DraftKings only
con = duckdb_con()
signals = con.execute(f"""
    SELECT *
    FROM read_parquet('s3://{S3_BUCKET}/{S3_SIGNALS}/*.parquet')
    WHERE bet_side = 'UNDER'
      AND bookmaker = '{BOOKMAKER}'
      AND (bookmaker_stale IS NULL OR bookmaker_stale = FALSE)
""").fetchdf()

# All signals (OVER+UNDER) for DraftKings — for prev-poll lookup
all_sigs = con.execute(f"""
    SELECT game_id, player_name, bookmaker, save_timestamp_utc, current_points, live_line
    FROM read_parquet('s3://{S3_BUCKET}/{S3_SIGNALS}/*.parquet')
    WHERE bookmaker = '{BOOKMAKER}'
""").fetchdf()
con.close()

signals['ts'] = pd.to_datetime(signals['save_timestamp_utc'], utc=True)
signals = signals.sort_values(['game_id','player_name','ts']).reset_index(drop=True)

all_sigs['ts'] = pd.to_datetime(all_sigs['save_timestamp_utc'], utc=True)
all_sigs = all_sigs.sort_values(['game_id','player_name','bookmaker','ts']).reset_index(drop=True)

grp = all_sigs.groupby(['game_id','player_name','bookmaker'])
all_sigs['prev_current_points'] = grp['current_points'].shift(1)
all_sigs['prev_live_line']      = grp['live_line'].shift(1)
all_sigs['prev_ts']             = grp['ts'].shift(1)
all_sigs['pts_delta']           = all_sigs['current_points'] - all_sigs['prev_current_points']
all_sigs['line_delta']          = all_sigs['live_line']      - all_sigs['prev_live_line']

prev_lookup = all_sigs.set_index(['game_id','player_name','bookmaker','ts'])[
    ['pts_delta','line_delta','prev_live_line','prev_ts']
]

print(f'DraftKings UNDER signals:  {len(signals):,}')
print(f'Unique games:               {signals["game_id"].nunique():,}')
print(f'Unique players:             {signals["player_name"].nunique():,}')
print(f'Date range: {signals["game_date_et"].min()} → {signals["game_date_et"].max()}')

## 2. Fetch ESPN PBP + Box Scores

In [ ]:
def fetch_3pt_makes(game_id):
    url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}'
    try:
        resp = SESSION.get(url, timeout=15); resp.raise_for_status()
    except Exception:
        return []
    makes = []
    for p in resp.json().get('plays', []):
        text = p.get('text','')
        wc   = p.get('wallclock','')
        if 'makes' not in text.lower(): continue
        if not any(x in text.lower() for x in ('three point','3-pt','three pointer')): continue
        if not wc: continue
        try: ts_utc = datetime.fromisoformat(wc.replace('Z','+00:00'))
        except Exception: continue
        makes.append({
            'player_name_lower': text.split(' makes ')[0].strip().lower(),
            'ts_utc': ts_utc,
            'period': p.get('period',{}).get('number', 0),
            'clock':  p.get('clock',{}).get('displayValue',''),
            'text':   text,
        })
    return makes

def fetch_final_scores(game_id):
    url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}'
    try:
        resp = SESSION.get(url, timeout=15); resp.raise_for_status()
    except Exception:
        return {}
    out = {}
    for team in resp.json().get('boxscore',{}).get('players',[]):
        for sb in team.get('statistics',[]):
            labels = sb.get('labels',[])
            try: pts_idx = labels.index('PTS')
            except ValueError: continue
            for ath in sb.get('athletes',[]):
                name = ath.get('athlete',{}).get('displayName','').lower()
                stats_ = ath.get('stats',[])
                if pts_idx < len(stats_):
                    try: out[name] = int(stats_[pts_idx])
                    except (ValueError, TypeError): pass
    return out

def normalize_name(name):
    return re.sub(r"[^a-z ]", "", name.lower()).strip()

def names_match(a, b):
    a, b = normalize_name(a), normalize_name(b)
    return a == b or a in b or b in a

print('ESPN functions defined.')

In [ ]:
game_ids = signals['game_id'].astype(str).unique().tolist()
print(f'Fetching ESPN data for {len(game_ids)} games...')

pbp_cache    = {}
scores_cache = {}

for i, gid in enumerate(game_ids):
    pbp_cache[gid]    = fetch_3pt_makes(gid)
    scores_cache[gid] = fetch_final_scores(gid)
    if (i+1) % 25 == 0 or (i+1) == len(game_ids):
        print(f'  {i+1}/{len(game_ids)} fetched')
    time.sleep(0.15)

print(f'Done. Total 3PT makes cached: {sum(len(v) for v in pbp_cache.values()):,}')

## 3. Tag Trigger Signals

In [ ]:
tagged_rows = []

for _, row in signals.iterrows():
    gid   = str(row['game_id'])
    pname = row['player_name']
    sig_ts = row['ts']

    pts_delta = line_delta = float('nan')
    prev_ts = pd.NaT
    try:
        prior = prev_lookup.loc[(gid, pname, BOOKMAKER, sig_ts)]
        if isinstance(prior, pd.DataFrame): prior = prior.iloc[0]
        pts_delta  = float(prior['pts_delta'])
        line_delta = float(prior['line_delta'])
        prev_ts    = prior['prev_ts']
    except KeyError:
        pass

    poll_interval_s = (sig_ts - pd.Timestamp(prev_ts)).total_seconds() if pd.notna(prev_ts) else float('nan')
    interval_clean  = (not np.isnan(poll_interval_s)) and (poll_interval_s <= MAX_POLL_INTERVAL)
    line_jumped     = (not np.isnan(line_delta)) and (line_delta >= MIN_LINE_JUMP)

    window_start = sig_ts - timedelta(seconds=WINDOW_SECONDS)
    trigger_make = None
    for m in pbp_cache.get(gid, []):
        if window_start <= m['ts_utc'] <= sig_ts and names_match(pname, m['player_name_lower']):
            trigger_make = m
            break

    is_trigger = (trigger_make is not None) and line_jumped and interval_clean

    scores = scores_cache.get(gid, {})
    final_pts = next((pts for name, pts in scores.items() if names_match(pname, name)), None)

    # OVER outcome: final > line → WIN; final < line → LOSS; final == line → PUSH (excluded)
    over_outcome = None
    if final_pts is not None and not pd.isna(row['live_line']):
        if final_pts > row['live_line']:  over_outcome = 'WIN'
        elif final_pts < row['live_line']: over_outcome = 'LOSS'
        # exact push: excluded from ROI calc

    tagged_rows.append({
        **row.to_dict(),
        'pts_delta':       pts_delta,
        'line_delta':      line_delta,
        'poll_interval_s': poll_interval_s,
        'prev_ts':         prev_ts,
        'trigger_ts_utc':  trigger_make['ts_utc'] if trigger_make else None,
        'interval_clean':  interval_clean,
        'line_jumped':     line_jumped,
        'is_trigger':      is_trigger,
        'trigger_period':  trigger_make['period'] if trigger_make else None,
        'trigger_clock':   trigger_make['clock']  if trigger_make else None,
        'trigger_text':    trigger_make['text']   if trigger_make else None,
        'final_pts':       final_pts,
        'over_outcome':    over_outcome,
        'under_outcome':   ('WIN' if final_pts < row['live_line'] else 'LOSS') if final_pts is not None and not pd.isna(row['live_line']) and final_pts != row['live_line'] else None,
    })

tagged   = pd.DataFrame(tagged_rows)
triggers = tagged[tagged['is_trigger']].copy()
evald    = (triggers[triggers['over_outcome'].notna()]
            .sort_values('ts')
            .drop_duplicates(subset=['game_id','player_name','live_line'], keep='first')
            .copy())

print(f'DraftKings UNDER signals:     {len(tagged):,}')
print(f'3PT trigger signals:           {len(triggers):,}')
print(f'Evaluated (deduped, outcome):  {len(evald):,}')
pushes = (triggers['over_outcome'].isna() & triggers['final_pts'].notna()).sum()
print(f'Pushes excluded:               {pushes}')

## Phase 1 — Foundational Validity

### 1.1 Actual OVER ROI using real `over_odds`

**Kill switch:** if real-odds OVER ROI < 0%, the edge is absorbed by juice. Stop here.

In [ ]:
def american_to_decimal(odds):
    odds = float(odds)
    return odds/100 + 1 if odds > 0 else 100/abs(odds) + 1

evald['decimal_over']  = evald['over_odds'].apply(american_to_decimal)
evald['decimal_under'] = evald['under_odds'].apply(american_to_decimal)

# Market hold per signal
evald['hold'] = (1/evald['decimal_over']) + (1/evald['decimal_under']) - 1

# OVER profit/loss per signal (flat $100 bets)
BET = 100
evald['over_profit'] = evald.apply(
    lambda r: (r['decimal_over'] - 1) * BET if r['over_outcome'] == 'WIN' else -BET, axis=1)

n      = len(evald)
wins   = (evald['over_outcome'] == 'WIN').sum()
profit = evald['over_profit'].sum()
roi    = profit / (n * BET)

print('=' * 55)
print(f'  PHASE 1.1 — OVER ROI (real over_odds)')
print('=' * 55)
print(f'  n signals:         {n}')
print(f'  W–L:               {wins}–{n-wins}')
print(f'  OVER hit rate:     {wins/n:.1%}')
print(f'  Profit ($100/bet): ${profit:+,.2f}')
print(f'  ROI:               {roi:+.1%}')
print()
n_neg = (evald['over_odds'] < 0).sum()
n_pos = (evald['over_odds'] > 0).sum()
print(f'  over_odds  — median: {evald["over_odds"].median():.0f}  (mean decimal: {evald["decimal_over"].mean():.4f})')
print(f'  Negative odds (OVER fav): {n_neg} ({n_neg/n:.0%})  |  Positive (OVER dog): {n_pos} ({n_pos/n:.0%})')
print(f'  Avg hold:          {evald["hold"].mean():.1%}')
print()

# At what OVER rate do we break even at actual mean odds?
mean_decimal_over = evald['decimal_over'].mean()
breakeven = 1 / mean_decimal_over
print(f'  Mean decimal OVER odds: {mean_decimal_over:.4f}')
print(f'  Breakeven OVER rate:    {breakeven:.1%}  (actual: {wins/n:.1%})')
verdict = 'PASS' if roi > 0.05 else ('MARGINAL' if roi > 0 else 'FAIL')
print(f'\n  Gate 1.1: {verdict}  (threshold: ROI > 5%)')

In [ ]:
# OVER odds distribution
print('=== over_odds distribution ===')
cuts   = [-float('inf'),-150,-130,-120,-115,-110,-105, 0, float('inf')]
labels = ['<-150','-150 to -130','-130 to -120','-120 to -115','-115 to -110','-110 to -105','-105 to 0','> 0 (positive)']
evald['odds_bucket'] = pd.cut(evald['over_odds'], bins=cuts, labels=labels)
dist = evald.groupby('odds_bucket', observed=True).agg(
    n=('over_outcome','count'),
    wins=('over_outcome', lambda x: (x=='WIN').sum()),
).assign(hit_rate=lambda d: d['wins']/d['n'])
print(dist.to_string())

### 1.2 Concentration Check — Is one player driving the effect?

In [ ]:
print('=== PHASE 1.2 — Signal Concentration ===')
player_stats = (
    evald.groupby('player_name')
    .agg(n=('over_outcome','count'),
         wins=('over_outcome', lambda x: (x=='WIN').sum()))
    .assign(hit_rate=lambda d: d['wins']/d['n'],
            pct_of_total=lambda d: d['n']/len(evald))
    .sort_values('n', ascending=False)
)

print(f'\nTop 15 players by signal count:')
print(player_stats.head(15).to_string())

top3_pct = player_stats['pct_of_total'].head(3).sum()
top1_pct = player_stats['pct_of_total'].iloc[0]
print(f'\nTop 1 player share: {top1_pct:.1%}')
print(f'Top 3 player share: {top3_pct:.1%}')
verdict = 'PASS' if top1_pct < 0.25 else 'FAIL'
print(f'Gate 1.2: {verdict}  (threshold: no single player > 25%)')

### 1.3 Market Hold Distribution

In [ ]:
print('=== PHASE 1.3 — Hold Distribution ===')
print(f'\nHold per signal (n={len(evald)}):')
print(evald['hold'].describe(percentiles=[.25,.5,.75,.9,.95]).apply(lambda x: f'{x:.2%}').to_string())
print(f'\nMean hold: {evald["hold"].mean():.2%}')
print(f'Signals with hold > 10%: {(evald["hold"] > 0.10).sum()} ({(evald["hold"] > 0.10).mean():.1%})')
print(f'Signals with hold > 15%: {(evald["hold"] > 0.15).sum()} ({(evald["hold"] > 0.15).mean():.1%})')

# Does OVER ROI survive after adjusting for mean hold?
print(f'\nHold-adjusted edge above breakeven: {wins/n - 1/mean_decimal_over:.1%}')

## Phase 2 — Effect Decomposition

### 2.1 Line Jump Magnitude → OVER Rate

In [ ]:
print('=== PHASE 2.1 — Line Jump Magnitude ===')
jump_cuts   = [0, 1.0, 2.0, 3.0, float('inf')]
jump_labels = ['+0.5–1.0', '+1.0–2.0', '+2.0–3.0', '+3.0+']
evald['jump_bucket'] = pd.cut(evald['line_delta'], bins=jump_cuts, labels=jump_labels)

jump_stats = (
    evald.groupby('jump_bucket', observed=True)
    .agg(n=('over_outcome','count'),
         wins=('over_outcome', lambda x: (x=='WIN').sum()),
         mean_jump=('line_delta','mean'))
    .assign(hit_rate=lambda d: d['wins']/d['n'],
            roi=lambda d: d.apply(
                lambda r: (evald[evald['jump_bucket']==r.name]['over_profit'].sum() /
                           (r['n']*BET)) if r['n'] > 0 else float('nan'), axis=1))
)

# Wilson 90% CI per bucket
for bkt in jump_stats.index:
    sub = evald[evald['jump_bucket'] == bkt]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    if n_ > 0:
        lo, hi = wilson_ci(k_, n_, alpha=0.10)
        jump_stats.loc[bkt, 'ci_lo'] = lo
        jump_stats.loc[bkt, 'ci_hi'] = hi

print(jump_stats.to_string())
print()
print('Interpretation: monotonically rising hit_rate → jump size predicts OVER edge')

### 2.2 MC UNDER Edge as Contrarian Indicator

In [ ]:
print('=== PHASE 2.2 — MC Edge as Contrarian Indicator ===')

# Tercile split by model UNDER edge
evald['edge_tercile'] = pd.qcut(evald['edge_after'], q=3, labels=['Low edge','Mid edge','High edge'])

tercile_stats = (
    evald.groupby('edge_tercile', observed=True)
    .agg(n=('over_outcome','count'),
         wins=('over_outcome', lambda x: (x=='WIN').sum()),
         mean_edge=('edge_after','mean'))
    .assign(hit_rate=lambda d: d['wins']/d['n'])
)

for bkt in tercile_stats.index:
    sub = evald[evald['edge_tercile'] == bkt]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    if n_ > 0:
        lo, hi = wilson_ci(k_, n_, alpha=0.10)
        tercile_stats.loc[bkt, 'ci_lo'] = lo
        tercile_stats.loc[bkt, 'ci_hi'] = hi

print(tercile_stats.to_string())
print()

# Pearson correlation: edge_after vs over_outcome (WIN=1)
evald['over_win_bin'] = (evald['over_outcome'] == 'WIN').astype(int)
r, pval = stats.pearsonr(evald['edge_after'], evald['over_win_bin'])
print(f'Pearson r(model_under_edge, over_win): {r:.3f}  p={pval:.3f}')
print()
if r > 0.05 and pval < 0.10:
    print('CONFIRMED: Higher MC UNDER edge predicts more OVER wins — model is a reliable contrarian indicator')
else:
    print('INCONCLUSIVE/ABSENT: MC edge not reliably anti-correlated with OVER outcomes')

### 2.3 Hot-Hand Stacking — Prior 3PT Count in Game

In [ ]:
print('=== PHASE 2.3 — Hot-Hand Stacking ===')

prior_3pt_counts = []
for _, row in evald.iterrows():
    gid    = str(row['game_id'])
    pname  = row['player_name']
    sig_ts = row['ts']
    makes  = pbp_cache.get(gid, [])
    # Count 3PT makes by this player BEFORE the trigger timestamp
    prior_count = sum(
        1 for m in makes
        if m['ts_utc'] < row['trigger_ts_utc']   # strictly before THIS trigger make
        and names_match(pname, m['player_name_lower'])
    )
    prior_3pt_counts.append(prior_count)

evald['prior_3pts'] = prior_3pt_counts

print('Prior 3PT makes in game before trigger:')
print(evald['prior_3pts'].value_counts().sort_index().to_string())
print()

evald['prior_bucket'] = evald['prior_3pts'].apply(
    lambda x: '0 (1st 3PT)' if x == 0 else ('1 (2nd 3PT)' if x == 1 else '2+ (3rd+ 3PT)'))

hot_stats = (
    evald.groupby('prior_bucket')
    .agg(n=('over_outcome','count'),
         wins=('over_outcome', lambda x: (x=='WIN').sum()))
    .assign(hit_rate=lambda d: d['wins']/d['n'])
)

for bkt in hot_stats.index:
    sub = evald[evald['prior_bucket'] == bkt]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    if n_ > 0:
        lo, hi = wilson_ci(k_, n_, alpha=0.10)
        hot_stats.loc[bkt, 'ci_lo'] = lo
        hot_stats.loc[bkt, 'ci_hi'] = hi

print(hot_stats.to_string())
print()
print('Interpretation: rising hit_rate with prior_count → hot hand compounds, 2nd+ 3PT is stronger signal')

### 2.4 Quarter Effect — Wilson Confidence Intervals

In [ ]:
print('=== PHASE 2.4 — Quarter Effect with CIs ===')

quarter_stats = []
for period in sorted(evald['trigger_period'].dropna().unique()):
    sub = evald[evald['trigger_period'] == period]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    lo, hi = wilson_ci(k_, n_, alpha=0.10)
    roi_ = sub['over_profit'].sum() / (n_ * BET)
    quarter_stats.append({
        'quarter':   f'Q{int(period)}',
        'n':         n_,
        'wins':      k_,
        'hit_rate':  f'{k_/n_:.1%}',
        'roi':       f'{roi_:+.1%}',
        'ci_90_lo':  f'{lo:.1%}',
        'ci_90_hi':  f'{hi:.1%}',
        'actionable': 'YES' if lo > 0.55 else ('MARGINAL' if lo > 0.52 else 'NO'),
    })

print(pd.DataFrame(quarter_stats).to_string(index=False))
print()
print('Actionable = CI 90% lower bound > 55%. Marginal = > 52%.')
print()

# Q4 late vs early
q4 = evald[evald['trigger_period'] == 4].copy()
if len(q4):
    def clock_to_seconds(clock_str):
        try:
            parts = str(clock_str).split(':')
            return int(parts[0])*60 + float(parts[1])
        except Exception:
            return float('nan')
    q4['secs_remaining'] = q4['trigger_clock'].apply(clock_to_seconds)
    q4['q4_phase'] = q4['secs_remaining'].apply(
        lambda s: 'Q4 early (>3 min)' if s > 180 else 'Q4 late (≤3 min)' if not np.isnan(s) else 'unknown')
    print('Q4 breakdown by time remaining:')
    for phase, sub in q4.groupby('q4_phase'):
        n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
        if n_ > 0:
            lo, hi = wilson_ci(k_, n_, alpha=0.10)
            print(f'  {phase}: n={n_}  hit={k_/n_:.1%}  CI=[{lo:.1%}, {hi:.1%}]')

## Phase 3 — Out-of-Sample Validity

### 3.1 Temporal Split: Regular Season vs Playoffs

In [ ]:
print('=== PHASE 3.1 — Temporal Split ===')

# NBA 2025-26 regular season ended ~April 13; playoffs started April 19
RS_END    = pd.Timestamp('2026-04-14', tz='UTC')
PO_START  = pd.Timestamp('2026-04-19', tz='UTC')

evald['regime'] = evald['ts'].apply(
    lambda t: 'Regular Season' if t < RS_END else ('Playoffs' if t >= PO_START else 'Gap/Playin'))

print('Signal distribution by regime:')
print(evald['regime'].value_counts().to_string())
print()

for regime, sub in evald.groupby('regime'):
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    if n_ == 0: continue
    roi_ = sub['over_profit'].sum() / (n_ * BET)
    lo, hi = wilson_ci(k_, n_, alpha=0.10)
    print(f'{regime}:  n={n_}  hit={k_/n_:.1%}  roi={roi_:+.1%}  CI=[{lo:.1%}, {hi:.1%}]')

print()
verdict = 'PASS' if all(
    (sub := evald[evald['regime']==r]) is not None and
    (sub['over_outcome']=='WIN').mean() > 0.55
    for r in ['Regular Season','Playoffs'] if len(evald[evald['regime']==r]) > 10
) else 'FAIL — check per-regime details above'
print(f'Gate 3.1: {verdict}')

### 3.2 Market Adaptation — Are Line Jumps Growing Over Time?

In [ ]:
print('=== PHASE 3.2 — Market Adaptation ===')

evald['week'] = evald['ts'].dt.to_period('W').apply(lambda p: p.start_time)

weekly = (
    evald.groupby('week')
    .agg(n=('over_outcome','count'),
         wins=('over_outcome', lambda x: (x=='WIN').sum()),
         mean_jump=('line_delta','mean'))
    .assign(hit_rate=lambda d: d['wins']/d['n'])
    .reset_index()
)

print('Weekly summary (mean line jump + OVER hit rate):')
print(weekly[['week','n','mean_jump','hit_rate']].to_string(index=False))
print()

# Spearman correlation: week number vs mean_jump (is jump growing?)
if len(weekly) > 3:
    weekly['week_num'] = range(len(weekly))
    r_jump, p_jump = stats.spearmanr(weekly['week_num'], weekly['mean_jump'])
    r_hit,  p_hit  = stats.spearmanr(weekly['week_num'], weekly['hit_rate'])
    print(f'Spearman r(week, mean_jump):  {r_jump:.3f}  p={p_jump:.3f}')
    print(f'Spearman r(week, hit_rate):   {r_hit:.3f}   p={p_hit:.3f}')
    print()
    if r_jump > 0.4 and p_jump < 0.15:
        print('WARNING: Line jumps appear to be growing over time — market may be adapting')
    else:
        print('OK: No clear upward trend in jump size')

## Summary Dashboard

In [ ]:
print('=' * 60)
print('  INVESTIGATION SUMMARY — DraftKings OVER After 3PT')
print('=' * 60)

n = len(evald)
wins = (evald['over_outcome']=='WIN').sum()
roi  = evald['over_profit'].sum() / (n * BET)
lo, hi = wilson_ci(wins, n, alpha=0.10)

print(f'\n  Signals (deduped):  {n}')
print(f'  W–L:                {wins}–{n-wins}')
print(f'  OVER hit rate:      {wins/n:.1%}  CI=[{lo:.1%}, {hi:.1%}]')
print(f'  Real-odds ROI:      {roi:+.1%}')
n_neg = (evald['over_odds'] < 0).sum()
n_pos = (evald['over_odds'] > 0).sum()
print(f'  Median over_odds:   {evald["over_odds"].median():.0f}  (mean decimal: {evald["decimal_over"].mean():.4f})')
print(f'  OVER favorite:      {n_neg} ({n_neg/n:.0%}) signals  |  OVER underdog: {n_pos} ({n_pos/n:.0%})')
print(f'  Mean hold:          {evald["hold"].mean():.1%}')
print(f'  Breakeven rate:     {1/evald["decimal_over"].mean():.1%}')
print()

print('  By quarter:')
for period in sorted(evald['trigger_period'].dropna().unique()):
    sub = evald[evald['trigger_period']==period]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    r_ = sub['over_profit'].sum() / (n_*BET)
    lo_, hi_ = wilson_ci(k_, n_, alpha=0.10)
    print(f'    Q{int(period)}: n={n_:3d}  hit={k_/n_:.1%}  roi={r_:+.1%}  CI=[{lo_:.1%}, {hi_:.1%}]')
print()

print('  By edge threshold:')
for thresh in EDGE_THRESHOLDS:
    sub = evald[evald['edge_after'] >= thresh]
    n_, k_ = len(sub), (sub['over_outcome']=='WIN').sum()
    r_ = sub['over_profit'].sum() / (n_*BET) if n_ else float('nan')
    print(f'    edge≥{thresh:.0%}: n={n_:3d}  hit={k_/n_:.1%}  roi={r_:+.1%}' if n_ else f'    edge≥{thresh:.0%}: n=0')